# ERA5 daily district weather extraction

Extracts daily ERA5-Land weather for the 25 canonical Sri Lankan districts.

This notebook is a **driver**. The extraction logic lives in
`scripts/5.extract_era5_daily.py` so that it is testable and version
controlled; the notebook only handles Colab authentication and runs it.
Editing the logic here instead of in the script would fork the two.

**Stage 1 of the climate pipeline.** Raw daily climate only. No reporting
periods, no lag features, no dengue merge. See
`docs/climate_dataset_schema.md` for the full specification.

## Output

`data/raw/climate_daily_district.csv` — one row per `date` x `node_id`.

## Runtime

The full 2005-2026 extraction is roughly 7 400 days x 25 districts. Earth
Engine is queried one year at a time and each completed year is cached to
`data/raw/era5_chunks/`, so an interrupted run resumes rather than
restarting. Expect a long session; start with the single-year smoke test
below before committing to the whole range.

## 1. Environment

On Colab, install the Earth Engine client and mount the repository. Skip
this cell when running locally against an existing checkout.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install --quiet earthengine-api

    from google.colab import drive

    drive.mount("/content/drive")

    # Point this at your checkout inside Drive.
    PROJECT_DIR = "/content/drive/MyDrive/AEGIS-Dengue"
else:
    from pathlib import Path

    PROJECT_DIR = str(Path.cwd().parent)

print("Project directory:", PROJECT_DIR)

## 2. Authenticate Earth Engine

Opens a consent flow the first time. Supply your Cloud project id — Earth
Engine requires one for new accounts.

In [ ]:
import ee

EE_PROJECT = ""  # e.g. "aegis-dengue-1234"

try:
    ee.Initialize(project=EE_PROJECT) if EE_PROJECT else ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT) if EE_PROJECT else ee.Initialize()

print("Earth Engine ready.")

## 3. Load the extraction module

The script filename starts with a digit, so it is loaded by path rather
than imported by name.

In [ ]:
import importlib.util
from pathlib import Path

script_path = Path(PROJECT_DIR) / "scripts" / "5.extract_era5_daily.py"

spec = importlib.util.spec_from_file_location("era5", script_path)
era5 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(era5)

print("ERA5 asset:  ", era5.ERA5_ASSET)
print("GADM level 1:", era5.GADM_LEVEL1_ASSET)
print("Version:     ", era5.EXTRACTION_VERSION)

## 4. Verify the district registry and boundaries

Confirms 25 polygons, and that every GADM name maps **explicitly** to a
canonical district with one `node_id` each. No fuzzy matching: an unknown
name raises rather than being guessed at.

In [ ]:
nodes = era5.load_nodes()

print(f"nodes.csv: {len(nodes)} districts, node_id 0-24\n")

polygons, source_level = era5.load_district_polygons(ee, nodes)

print(f"Boundaries: {source_level}")
print(f"Polygons:   {polygons.size().getInfo()}")
print("\nDistrict mapping verified: 25 polygons, 25 node ids, 25 names.")

nodes.head()

## 5. Smoke test: one week

Extract a short window first and check the values are physically sensible.
This catches the two errors that are otherwise expensive to discover after
a full run:

- **rainfall off by 1000x** — total precipitation is a running accumulation,
  so it must be read as a daily total, not summed from raw steps;
- **temperature near -273** — Kelvin conversion applied twice.

Expect tropical values: 24-32 C, humidity 65-90%, rainfall 0-50 mm.

In [ ]:
from datetime import date

sample = era5.finalise_chunk(
    era5.extract_year(
        ee, polygons, 2015, date(2015, 6, 1), date(2015, 6, 7)
    )
)

print(f"rows: {len(sample)}  (expect 7 x 25 = 175)\n")

sample[
    [
        "rainfall_mm",
        "temperature_mean_c",
        "temperature_min_c",
        "temperature_max_c",
        "relative_humidity_mean",
        "wind_speed_mean",
    ]
].describe().round(2)

In [ ]:
# Stop here unless these hold.
assert sample["temperature_mean_c"].between(0, 50).all(), \
    "Temperature outside 0-50 C: check the Kelvin conversion."

assert sample["rainfall_mm"].between(0, 1000).all(), \
    "Rainfall outside 0-1000 mm: precipitation is an accumulation, not a sum."

assert (
    sample["temperature_min_c"] <= sample["temperature_mean_c"]
).all(), "min exceeds mean."

assert (
    sample["temperature_mean_c"] <= sample["temperature_max_c"]
).all(), "mean exceeds max."

assert (
    sample["dewpoint_mean_c"] <= sample["temperature_mean_c"] + 0.5
).all(), "Dewpoint above air temperature is impossible."

print("Smoke test passed. Values are physically plausible.")

## 6. Resolve the full date range

Covers every dengue reporting period plus 52 weeks of prior history, so the
earliest forecastable period has a full lag window behind it.

ERA5-Land trails real time by roughly three months. If the dengue series
extends past what is published, the cell below says so rather than
truncating silently.

In [ ]:
start, end = era5.resolve_date_range(None, None)

expected_days = (end - start).days + 1

print(f"Start:         {start}")
print(f"End:           {end}")
print(f"Days:          {expected_days}")
print(f"Expected rows: {expected_days * 25}")
print(f"Chunks:        {end.year - start.year + 1} years")

## 7. Extract in yearly chunks

Each year is cached to `data/raw/era5_chunks/`. Re-running skips completed
years, so an interrupted session resumes where it stopped.

In [ ]:
era5.CHUNK_DIR.mkdir(parents=True, exist_ok=True)

for year in range(start.year, end.year + 1):
    path = era5.chunk_path(year)

    if path.exists():
        print(f"  {year}: cached, skipping")
        continue

    print(f"  {year}: extracting...", end=" ", flush=True)

    chunk = era5.finalise_chunk(
        era5.extract_year(ee, polygons, year, start, end)
    )

    chunk.to_csv(path, index=False)

    print(f"{len(chunk)} rows")

print("\nAll chunks present.")

## 8. Combine into the single output

In [ ]:
combined = era5.combine_chunks(start, end)

era5.RAW_DIR.mkdir(parents=True, exist_ok=True)
combined.to_csv(era5.OUTPUT_PATH, index=False)

print(f"Wrote {era5.OUTPUT_PATH}")
print(f"Rows:  {len(combined)}")

combined.head()

## 9. Validation

Prints the required report: date bounds, unique dates, district count, row
count, duplicate district-dates, and missing values by column. Findings are
reported explicitly rather than raised, so a short or gapped series is
visible in full.

In [ ]:
summary = era5.validate_extraction(combined, nodes, start, end)

era5.write_manifest(summary, start, end, source_level)

## 10. Sanity plot

Colombo rainfall and temperature. The two monsoons should be visible: the
southwest monsoon around May-September and the northeast around
December-February. A flat or noise-like series means the aggregation is
wrong, most likely sampling ocean pixels.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

colombo = combined.loc[combined["canonical_name"].eq("Colombo")].copy()
colombo["date"] = pd.to_datetime(colombo["date"])
colombo = colombo.set_index("date").sort_index()

monthly = colombo.resample("ME").agg(
    {"rainfall_mm": "sum", "temperature_mean_c": "mean"}
)

figure, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].bar(monthly.index, monthly["rainfall_mm"], width=20)
axes[0].set_ylabel("Rainfall (mm/month)")
axes[0].set_title("Colombo — monthly rainfall and mean temperature")

axes[1].plot(monthly.index, monthly["temperature_mean_c"])
axes[1].set_ylabel("Mean temperature (C)")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

## Next stage

`data/raw/climate_daily_district.csv` is now raw daily climate data,
independent of the dengue dataset.

**Deliberately not done here** — these belong to later stages:

- aggregating to reporting periods (`period_id`);
- lag features and rolling statistics;
- merging dengue cases.

Stage 2 joins these daily rows to `data/interim/reporting_calendar.csv` on
the closed interval `start_date` to `end_date`. Note that rainfall **sums**
across a period while temperatures take min/mean/max, and that the 2009
periods are 8 and 6 days long rather than 7 — so summed rainfall needs
`reporting_days` alongside it to stay comparable.